In [1]:
import numpy as np
import pandas as pd
from pyspark.sql import SparkSession
from pyspark.sql.functions import monotonically_increasing_id
import pyspark.pandas as ps

import os

/Users/danilsamsutdinov/HypEx/.venv/lib/python3.11/site-packages/pyspark/pandas/__init__.py:50: UserWarning: 'PYARROW_IGNORE_TIMEZONE' environment variable was not set. It is required to set this environment variable to '1' in both driver and executor sides if you use pyarrow>=2.0.0. pandas-on-Spark will set it for you but it does not work if there is a Spark context already launched.
  warnings.warn(


In [2]:
from hypex.matching import Matching
from hypex.ml.faiss import FaissNearestNeighbors
from hypex.transformers import TypeCaster
from hypex.dataset import Dataset, InfoRole, TreatmentRole, FeatureRole, TargetRole, ExperimentData, AdditionalMatchingRole, AdditionalStatisticRole, DisabledRole
from hypex.utils import BackendsEnum
from hypex.experiments import Experiment, OnRoleExperiment
from hypex.comparators import MahalanobisDistance
from hypex.encoders.encoders import DummyEncoder
from hypex.comparators import TTest, Chi2Test
from hypex.comparators.distances import MahalanobisDistance
from hypex.operators import Bias, MatchingMetrics
from hypex.analyzers import MatchingAnalyzer

In [3]:
# --- 1. Настройки окружения для macOS (Важно!) ---
# На macOS иногда возникают проблемы с форком процессов Java (Executor'ы не стартуют).
# Эта переменная часто решает проблему "Connection refused" или краши при запуске local-cluster
os.environ['OBJC_DISABLE_INITIALIZE_FORK_SAFETY'] = 'YES'

# Очистка старых сессий и переменных (как у вас было)
try:
    existing_spark = SparkSession.getActiveSession()
    if existing_spark:
        existing_spark.stop()
        print("✅ Существующая сессия остановлена.")
except:
    pass

for key in list(os.environ.keys()):
    if 'SPARK' in key or 'JAVA_OPTS' in key:
        del os.environ[key]

# --- 2. Конфигурация Кластера ---
# Формат: local-cluster[число_воркеров, ядер_на_воркер, память_на_воркер_в_МБ]
# Мы просим 2 экзекутора, по 1 ядру, по 2 ГБ памяти каждый
NUM_EXECUTORS = 2
CORES_PER_EXECUTOR = 4
MEMORY_PER_EXECUTOR_MB = 2048 

MASTER_URL = f"local-cluster[{NUM_EXECUTORS}, {CORES_PER_EXECUTOR}, {MEMORY_PER_EXECUTOR_MB}]"

print(f"🚀 Запуск в режиме: {MASTER_URL}")

sp_s = (SparkSession.builder
    .master(MASTER_URL)
    .appName("LocalClusterTest")
    # Память драйвера (остается у вас)
    .config("spark.driver.memory", "2g") 
    # Память экзекутора (должна соответствовать или быть меньше чем в master URL)
    .config("spark.executor.memory", "2g")
    .config("spark.executor.cores", "4")
    .config("spark.executor.instances", NUM_EXECUTORS)
    # Увеличиваем память под оверхед, чтобы избежать ошибок выделения памяти
    .config("spark.memory.fraction", "0.6")
    .config("spark.sql.shuffle.partitions", "4") # Для тестов меньше дефолтных 200
    .getOrCreate()
)

sp_s.sparkContext.setLogLevel("WARN")

# --- 3. Проверка конфигурации ---
print(f"✅ Сессия создана.")
print(f"Driver Memory Config: {sp_s.conf.get('spark.driver.memory')}")
print(f"Executor Memory Config: {sp_s.conf.get('spark.executor.memory')}")

# Проверка количества экзекуторов (может занять пару секунд на старт)
import time
time.sleep(3) 
num_executors = len(sp_s.sparkContext.parallelize(range(10), NUM_EXECUTORS).glom().collect())
print(f"📊 Активных экзекуторов (проверка через RDD): {num_executors}")

# --- 4. Тест на распределение (Пример) ---
# Чтобы убедиться, что задача ушла на экзекуторы, а не осталась на драйвере
def print_executor_info(iterator):
    import os
    # Получаем ID экзекутора из переменных окружения процесса
    executor_id = os.environ.get('SPARK_EXECUTOR_ID', 'Driver/Local')
    process_id = os.getpid()
    return [f"Executor ID: {executor_id}, PID: {process_id}"]

# Создаем датафрейм и применяем трансформацию
df = sp_s.range(0, 10, 1, 4) # 4 партиции
result = df.rdd.mapPartitions(print_executor_info).collect()

print("\n🖥️ Где выполнялись задачи:")
for line in result:
    print(line)

# Не забывайте останавливать сессию в конце скрипта, так как процессы тяжелые
# sp_s.stop() 

🚀 Запуск в режиме: local-cluster[2, 4, 2048]


26/06/26 20:40:34 WARN Utils: Your hostname, MacBook-Pro-Danil.local resolves to a loopback address: 127.0.0.1; using 10.246.121.44 instead (on interface en0)
26/06/26 20:40:34 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/06/26 20:40:34 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


✅ Сессия создана.
Driver Memory Config: 2g
Executor Memory Config: 2g


📊 Активных экзекуторов (проверка через RDD): 2

🖥️ Где выполнялись задачи:
Executor ID: Driver/Local, PID: 98731
Executor ID: Driver/Local, PID: 98730
Executor ID: Driver/Local, PID: 98735
Executor ID: Driver/Local, PID: 98734


In [ ]:
sp_s

In [4]:
n = 5000  # увеличьте для теста IVF-индексов
df = pd.DataFrame({
    "treatment": np.random.choice([0, 1], size=n, p=[0.6, 0.4]),
    "feat_num_1": np.random.normal(loc=10, scale=3, size=n),
    "feat_num_2": np.random.normal(loc=-2, scale=1.5, size=n),
    "feat_cat": np.random.choice(["A", "B", "C"], size=n),
    "target": np.random.normal(loc=100, scale=10, size=n)
})

In [5]:
nn = 200
index_df = pd.DataFrame(
    {
        '0': np.random.randint(0, 5000, nn),
        '1': np.random.randint(0, 5000, nn),
        '2': np.random.randint(0, 5000, nn),
        '3': np.random.randint(0, 5000, nn),
        '4': np.random.randint(0, 5000, nn),
        'group': [0] * (nn//4) + [1] * (nn//4) + [2] * (nn//4) + [3] * (nn//4)
    }
)

# index_df = pd.concat([index_df, pd.DataFrame(data={
#     '0': np.nan,
#     '1': np.nan,
#     '2': np.nan,
#     '3': np.nan,
#     '4': np.nan,
#     'group': np.nan
# }, index=[0])]).reset_index(drop=True)
# index_df

In [6]:
session = (
            SparkSession.builder
            .master("local[*]")
            .config("spark.driver.memory", "4g")
            .config("spark.executor.memory", "4g")
            .config("spark.memory.fraction", "0.8") 
            .config("spark.memory.storageFraction", "0.3")
            # .config("spark.jars.packages", "ch.cern.sparkmeasure:spark-measure_2.12:0.23") 
            .getOrCreate()
          )

26/06/26 20:40:40 WARN SparkSession: Using an existing Spark session; only runtime SQL configurations will take effect.


In [7]:
index_ds = Dataset(
    roles={
        'group': TargetRole()
    },
    # data=index_df
    data=session.createDataFrame(index_df),
    # data=sp_s.createDataFrame(index_df),
    session=session,
    backend=BackendsEnum.pandas
    # session=sp_s
)

index_ds

,0,1,2,3,4,group
0,1122,1698,867,2897,1968,0
1,1737,1417,4492,71,2881,0
2,4529,1989,1360,1854,2091,0
3,1745,2610,4036,1169,4957,0
4,3715,1435,1317,3853,2113,0
...,...,...,...,...,...,...
195,2221,810,2597,2408,3360,3
196,4663,1540,4306,1380,1511,3
197,3475,4581,372,3177,1297,3
198,476,1042,1598,3433,2038,3


In [8]:
round(index_ds)

,0,1,2,3,4,group
0,1122,1698,867,2897,1968,0
1,1737,1417,4492,71,2881,0
2,4529,1989,1360,1854,2091,0
3,1745,2610,4036,1169,4957,0
4,3715,1435,1317,3853,2113,0
...,...,...,...,...,...,...
195,2221,810,2597,2408,3360,3
196,4663,1540,4306,1380,1511,3
197,3475,4581,372,3177,1297,3
198,476,1042,1598,3433,2038,3


In [9]:
roles = {
    "treatment": TreatmentRole(),
    "feat_num_1": FeatureRole(),
    "feat_num_2": FeatureRole(),
    "feat_cat": FeatureRole(str),
    "target": TargetRole(),
    # "index": FeatureRole()  # индекс тоже должен быть в ролях, чтобы не отфильтровался
}

dataset = Dataset(
    roles=roles,
    data=df,
    backend=BackendsEnum.pandas,
)

matcher = Matching(distance="mahalanobis", n_neighbors=2, quality_tests=["t-test", "chi2-test"])
result = matcher.execute(dataset)

/Users/danilsamsutdinov/HypEx/hypex/dataset/backends/pandas_backend.py:1177: FutureWarning: Calling int on a single element Series is deprecated and will raise a TypeError in the future. Use int(ser.iloc[0]) instead
  return int(self.data[group_cols].nunique())
/Users/danilsamsutdinov/HypEx/hypex/dataset/backends/pandas_backend.py:1177: FutureWarning: Calling int on a single element Series is deprecated and will raise a TypeError in the future. Use int(ser.iloc[0]) instead
  return int(self.data[group_cols].nunique())
/Users/danilsamsutdinov/HypEx/hypex/dataset/backends/pandas_backend.py:1177: FutureWarning: Calling int on a single element Series is deprecated and will raise a TypeError in the future. Use int(ser.iloc[0]) instead
  return int(self.data[group_cols].nunique())
/Users/danilsamsutdinov/HypEx/hypex/dataset/backends/pandas_backend.py:1472: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.

[DEBUG MatchingMetrics] compare_result type: <class 'dict'>
[DEBUG MatchingMetrics] compare_result keys/shape: dict_keys(['ATT', 'ATC', 'ATE'])
[DEBUG MatchingMetrics] compare_result content: {'ATT': [0.15194236878871514, 0.2676868456552965, 0.5702981958689239, -0.372723848695666, 0.6766085862730963], 'ATC': [0.18519592113501304, 0.6510119047339038, 0.776047202751087, -1.0907874121434382, 1.4611792544134645], 'ATE': [0.17229354282464945, 0.39066933031809226, 0.6591974900081565, -0.5934183445988114, 0.9380054302481103]}
[DEBUG MatchingAnalyzer] variables type: <class 'dict'>
[DEBUG MatchingAnalyzer] variables content: {'ATT': [0.15194236878871514, 0.2676868456552965, 0.5702981958689239, -0.372723848695666, 0.6766085862730963], 'ATC': [0.18519592113501304, 0.6510119047339038, 0.776047202751087, -1.0907874121434382, 1.4611792544134645], 'ATE': [0.17229354282464945, 0.39066933031809226, 0.6591974900081565, -0.5934183445988114, 0.9380054302481103]}
[DEBUG MatchingAnalyzer] before transpose 

/Users/danilsamsutdinov/HypEx/hypex/dataset/backends/pandas_backend.py:1177: FutureWarning: Calling int on a single element Series is deprecated and will raise a TypeError in the future. Use int(ser.iloc[0]) instead
  return int(self.data[group_cols].nunique())
/Users/danilsamsutdinov/HypEx/hypex/comparators/abstract.py:242: UserWarning: baseline_field_data must have only one column when the comparison is done by matched_pairs. 2 passed. FaissNearestNeighbors┴┴┴0 will be used.
  warnings.warn(
/Users/danilsamsutdinov/HypEx/hypex/comparators/abstract.py:242: UserWarning: baseline_field_data must have only one column when the comparison is done by matched_pairs. 2 passed. FaissNearestNeighbors┴┴┴0 will be used.
  warnings.warn(
/Users/danilsamsutdinov/HypEx/hypex/comparators/abstract.py:242: UserWarning: baseline_field_data must have only one column when the comparison is done by matched_pairs. 2 passed. FaissNearestNeighbors┴┴┴0 will be used.
  warnings.warn(
/Users/danilsamsutdinov/Hyp

In [10]:
result.resume

,Effect Size,Standard Error,P-value,CI Lower,CI Upper
ATT,0.1519,0.2677,0.5703,-0.3727,0.6766
ATC,0.1852,0.6510,0.7760,-1.0908,1.4612
ATE,0.1723,0.3907,0.6592,-0.5934,0.9380


In [11]:
roles = {
    "treatment": TreatmentRole(),
    "feat_num_1": FeatureRole(),
    "feat_num_2": FeatureRole(),
    "feat_cat": FeatureRole(str),
    "target": TargetRole(),
    # "index": FeatureRole()  # индекс тоже должен быть в ролях, чтобы не отфильтровался
}

dataset = Dataset(
    roles=roles,
    data=df,
    backend=BackendsEnum.pandas,
)

In [12]:
roles = {
    "treatment": TreatmentRole(),
    "feat_num_1": FeatureRole(),
    "feat_num_2": FeatureRole(),
    "feat_cat": FeatureRole(str),
    "target": TargetRole(),
    # "index": FeatureRole()  # индекс тоже должен быть в ролях, чтобы не отфильтровался
}

dataset = Dataset(
    roles=roles,
    data=df,
    backend=BackendsEnum.spark,
    session=sp_s,
)

matcher = Matching(distance="mahalanobis", n_neighbors=2, quality_tests=["t-test", "chi2-test"])
result = matcher.execute(dataset)

/Users/danilsamsutdinov/HypEx/.venv/lib/python3.11/site-packages/pyspark/pandas/utils.py:1016: PandasAPIOnSparkAdviceWarning: If `index_col` is not specified for `to_spark`, the existing index is lost when converting to Spark DataFrame.
  warnings.warn(message, PandasAPIOnSparkAdviceWarning)
/Users/danilsamsutdinov/HypEx/.venv/lib/python3.11/site-packages/pyspark/pandas/utils.py:1016: PandasAPIOnSparkAdviceWarning: If `index_col` is not specified for `to_spark`, the existing index is lost when converting to Spark DataFrame.
  warnings.warn(message, PandasAPIOnSparkAdviceWarning)
/Users/danilsamsutdinov/HypEx/.venv/lib/python3.11/site-packages/pyspark/pandas/utils.py:1016: PandasAPIOnSparkAdviceWarning: If `index_col` is not specified for `to_spark`, the existing index is lost when converting to Spark DataFrame.
  warnings.warn(message, PandasAPIOnSparkAdviceWarning)
26/06/26 20:40:46 WARN AttachDistributedSequenceExec: clean up cached RDD(34) in AttachDistributedSequenceExec(178)
26/06

[DEBUG MatchingMetrics] compare_result type: <class 'dict'>
[DEBUG MatchingMetrics] compare_result keys/shape: dict_keys(['ATT', 'ATC', 'ATE'])
[DEBUG MatchingMetrics] compare_result content: {'ATT': [0.14790014682862093, 0.267721665254657, 0.5806469570760195, -0.37683431707050674, 0.6726346107277487], 'ATC': [0.18418096048311272, 0.6511225715583033, 0.777278942718922, -1.0920192797711619, 1.460381200737387], 'ATE': [0.1701040047851699, 0.39070836063088543, 0.6632914568476194, -0.5956843820513655, 0.9358923916217052]}
[DEBUG MatchingAnalyzer] variables type: <class 'dict'>
[DEBUG MatchingAnalyzer] variables content: {'ATT': [0.14790014682862093, 0.267721665254657, 0.5806469570760195, -0.37683431707050674, 0.6726346107277487], 'ATC': [0.18418096048311272, 0.6511225715583033, 0.777278942718922, -1.0920192797711619, 1.460381200737387], 'ATE': [0.1701040047851699, 0.39070836063088543, 0.6632914568476194, -0.5956843820513655, 0.9358923916217052]}
[DEBUG MatchingAnalyzer] before transpose sh

/Users/danilsamsutdinov/HypEx/.venv/lib/python3.11/site-packages/pyspark/pandas/utils.py:1016: PandasAPIOnSparkAdviceWarning: If `index_col` is not specified for `to_spark`, the existing index is lost when converting to Spark DataFrame.
  warnings.warn(message, PandasAPIOnSparkAdviceWarning)
/Users/danilsamsutdinov/HypEx/.venv/lib/python3.11/site-packages/pyspark/pandas/utils.py:1016: PandasAPIOnSparkAdviceWarning: If `index_col` is not specified for `to_spark`, the existing index is lost when converting to Spark DataFrame.
  warnings.warn(message, PandasAPIOnSparkAdviceWarning)
/Users/danilsamsutdinov/HypEx/.venv/lib/python3.11/site-packages/pyspark/pandas/utils.py:1016: PandasAPIOnSparkAdviceWarning: If `index_col` is not specified for `to_spark`, the existing index is lost when converting to Spark DataFrame.
  warnings.warn(message, PandasAPIOnSparkAdviceWarning)
/Users/danilsamsutdinov/HypEx/.venv/lib/python3.11/site-packages/pyspark/pandas/utils.py:1016: PandasAPIOnSparkAdviceWarn

[DEBUG MatchingReporter] analyzer_id: MatchingAnalyzer┴┴
[DEBUG MatchingReporter] result_ds type: <class 'hypex.dataset.dataset.SmallDataset'>
[DEBUG MatchingReporter] result_ds roles: ['Effect Size', 'Standard Error', 'P-value', 'CI Lower', 'CI Upper']
[DEBUG MatchingReporter] result_ds columns: Index(['Effect Size', 'Standard Error', 'P-value', 'CI Lower', 'CI Upper'], dtype='object')
[DEBUG MatchingReporter] result_ds data empty? False
[DEBUG MatchingOutput] self.resume.is_empty(): False
[DEBUG MatchingOutput] self.resume.columns: Index(['Effect Size', 'Standard Error', 'P-value', 'CI Lower', 'CI Upper'], dtype='object')
[DEBUG MatchingOutput] self.resume.roles: ['Effect Size', 'Standard Error', 'P-value', 'CI Lower', 'CI Upper']
[DEBUG MatchingOutput] self.resume.data:
     Effect Size  Standard Error   P-value  CI Lower  CI Upper
ATT     0.147900        0.267722  0.580647 -0.376834  0.672635
ATC     0.184181        0.651123  0.777279 -1.092019  1.460381
ATE     0.170104        0.3

/Users/danilsamsutdinov/HypEx/.venv/lib/python3.11/site-packages/pyspark/pandas/utils.py:1016: PandasAPIOnSparkAdviceWarning: If `index_col` is not specified for `to_spark`, the existing index is lost when converting to Spark DataFrame.
  warnings.warn(message, PandasAPIOnSparkAdviceWarning)
/Users/danilsamsutdinov/HypEx/.venv/lib/python3.11/site-packages/pyspark/pandas/utils.py:1016: PandasAPIOnSparkAdviceWarning: `to_pandas` loads all data into the driver's memory. It should only be used if the resulting pandas DataFrame is expected to be small.
  warnings.warn(message, PandasAPIOnSparkAdviceWarning)
/Users/danilsamsutdinov/HypEx/.venv/lib/python3.11/site-packages/pyspark/pandas/utils.py:1016: PandasAPIOnSparkAdviceWarning: `to_list` loads all data into the driver's memory. It should only be used if the resulting list is expected to be small.
  warnings.warn(message, PandasAPIOnSparkAdviceWarning)
/Users/danilsamsutdinov/HypEx/.venv/lib/python3.11/site-packages/pyspark/pandas/utils.p

In [ ]:
"""
PANDAS case
"""
# 4. Обёртка в Dataset фреймворка
roles = {
    "treatment": TreatmentRole(),
    "feat_num_1": FeatureRole(),
    "feat_num_2": FeatureRole(),
    "feat_cat": FeatureRole(str),
    "target": TargetRole(),
    # "index": FeatureRole()  # индекс тоже должен быть в ролях, чтобы не отфильтровался
}

dataset = Dataset(
    roles=roles,
    data=df,
    backend=BackendsEnum.pandas,
)

pandas_experiment = Experiment(
    executors=[
        DummyEncoder(),
        MahalanobisDistance(
            grouping_role=TreatmentRole(),
            weights=None
        ),
        TypeCaster(
            dtype={int: float},
            roles=[FeatureRole(), TargetRole()],
        ),
        FaissNearestNeighbors(
            grouping_role=TreatmentRole(),
            two_sides=True,
            test_pairs=False,
            faiss_mode="auto",
            n_neighbors=2,
        ),
        Bias(
            grouping_role=TreatmentRole(), 
            target_roles=[TargetRole()]
        ),
        MatchingMetrics(
                grouping_role=TreatmentRole(),
                target_roles=[TargetRole()],
                metric="ate",
                n_neighbors=2,
        ),
        MatchingAnalyzer(),
        OnRoleExperiment(
            executors=[
                TTest(
                    grouping_role=TreatmentRole(),
                    compare_by="matched_pairs",
                    baseline_role=AdditionalMatchingRole(),
                ),
                Chi2Test(
                    grouping_role=TreatmentRole(),
                    compare_by="matched_pairs",
                    baseline_role=AdditionalMatchingRole()
                )
            ],
            role=FeatureRole()
        )
    ]
)
pandas_result = pandas_experiment.execute(ExperimentData(dataset))

In [ ]:
pandas_result.analysis_tables

In [ ]:
pandas_result.analysis_tables

In [ ]:
"""
PYSPARK case
"""
# 3. Конвертация в Spark + ОБЯЗАТЕЛЬНАЯ колонка `index` (требование Faiss)
spark_df = sp_s.createDataFrame(df)

# 4. Обёртка в Dataset фреймворка
roles = {
    "treatment": TreatmentRole(),
    "feat_num_1": FeatureRole(),
    "feat_num_2": FeatureRole(),
    "feat_cat": FeatureRole(str),
    "target": TargetRole(),
    # "index": FeatureRole()  # индекс тоже должен быть в ролях, чтобы не отфильтровался
}

dataset = Dataset(
    roles=roles,
    data=spark_df,
    # data=session.createDataFrame(df),
    # data=df,
    backend=BackendsEnum.spark,
    session=sp_s,
    # session=session
)

spark_experiment = Experiment(
    executors=[
        DummyEncoder(),
        MahalanobisDistance(
            grouping_role=TreatmentRole(),
            weights=None
        ),
        TypeCaster(
            dtype={int: float},
            roles=[FeatureRole(), TargetRole()],
        ),
        FaissNearestNeighbors(
            grouping_role=TreatmentRole(),
            two_sides=True,
            test_pairs=False,
            faiss_mode="auto",
            n_neighbors=2,
        ),
        Bias(
            grouping_role=TreatmentRole(), 
            target_roles=[TargetRole()]
        ),
        MatchingMetrics(
                grouping_role=TreatmentRole(),
                target_roles=[TargetRole()],
                metric="ate",
                n_neighbors=2,
        ),
        MatchingAnalyzer(),
        OnRoleExperiment(
            executors=[
                TTest(
                    grouping_role=TreatmentRole(),
                    compare_by="matched_pairs",
                    baseline_role=AdditionalMatchingRole(),
                ),
                Chi2Test(
                    grouping_role=TreatmentRole(),
                    compare_by="matched_pairs",
                    baseline_role=AdditionalMatchingRole()
                )
            ],
            role=FeatureRole()
        )
    ]
)
spark_result = spark_experiment.execute(ExperimentData(dataset))

In [ ]:
spark_result.analysis_tables

In [ ]:
spark_result.ds.unpersist()

In [ ]:
spark_result.variables["MahalanobisDistance┴┴['feat_num_1', 'feat_num_2', 'DummyEncoder||_feat_cat_B', 'DummyEncoder||_feat_cat_C']"]["['feat_num_1', 'feat_num_2', 'DummyEncoder┴┴_feat_cat_B', 'DummyEncoder┴┴_feat_cat_C']"]

In [ ]:
spark_result.field_search(AdditionalStatisticRole())

In [ ]:
spark_result.ds

In [ ]:
# for label, ds in result.variables['Bias┴┴[\'target\', \'target_matched\']'].items():
#     ds.to_small_dataset().data.to_csv(f"{label}.csv")

In [ ]:
spark_result.analysis_tables

In [ ]:
sp_s.stop()

In [ ]:
r = pd.read_csv('result_[5].csv', names=['index', 'value'])
r.dropna(subset=['index']).fillna(0)

In [ ]:

# 5. Настройка Matching
matching = Matching(
    distance="mahalanobis",
    # metric="ate",
    bias_estimation=False,      # отключаем для упрощения дебага
    quality_tests=["t-test"],   # только t-test для скорости
    faiss_mode="base",          # "base" → IndexFlatL2 (без IVF), проще отлаживать
    n_neighbors=1,
    encode_categories=True      # DummyEncoder включится автоматически
)

# 6. Запуск
print("🚀 Запуск пайплайна Matching...")
result_data = matching.execute(dataset)

print("✅ Выполнено успешно!")
print(f"📊 Additional Fields: {result_data.additional_fields.columns}")
print(f"📦 Groups Keys: {list(result_data.groups.keys())}")

In [ ]:
sp_s.stop()